# 02 - Upgrade experiments (research-grade)
This notebook reads the saved results of the upgrade phases and re-runs one small nested-Optuna example so you can see how the tuning is done. **Development data only** (films before 2023-01-01) is used for every experiment; the 2023+ test set was scored once in `hitflop.final`.

Run order used for the project: `python -m hitflop.pipeline_phase1`, `hitflop.phase3`, `hitflop.phase45`, `hitflop.final`, `hitflop.artifacts`, `hitflop.phase6`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from hitflop import config as C, metrics as M, tuning as T, validation as V, models as MD
from hitflop.world import build_world
pd.set_option('display.width', 200)
merged, lab, dev, test = build_world()
print(len(lab), 'labelled |', len(dev), 'development |', len(test), 'frozen test (not used below)')

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


854 labelled | 578 development | 276 frozen test (not used below)


## 1. Same cohort as the original notebook
854 labelled films, 578 development, 276 test; Hit = revenue / budget >= 2.0.

In [2]:
pd.read_csv(C.OUT_DIR/'phase1'/'audit_labelled_share_language.csv')

,language,movies,labelled,share
0,hi,1327,409,0.308214
1,kn,234,35,0.149573
2,ml,951,127,0.133544
3,ta,945,109,0.115344
4,te,742,174,0.234501


## 2. Does any new feature help? (development, out-of-fold)
Every comparison uses the identical validated films, a paired bootstrap, and Benjamini-Hochberg correction.

In [3]:
c = pd.read_csv(C.OUT_DIR/'phase3'/'3a_comparisons.csv')
print(int(c.convincing.sum()), 'of', len(c), 'comparisons are convincing')
c[['model','set','vs','diff','ci_low','ci_high','q_bh_all','fold_wins']].round(3)

0 of 48 comparisons are convincing


,model,set,vs,diff,ci_low,ci_high,q_bh_all,fold_wins
0,LR,B+,B,-0.020,-0.034,-0.007,0.120,1
1,LR,C+,C,0.003,-0.004,0.010,0.836,2
2,LR,D+,D,-0.014,-0.031,0.002,0.475,1
3,LR,A+CAL,A,-0.007,-0.032,0.018,0.887,2
4,LR,A+COMP,A,-0.009,-0.027,0.009,0.737,3
5,LR,A+CAL+COMP,A,-0.002,-0.029,0.027,0.971,1
6,LR,A+HIST,A,-0.052,-0.096,-0.006,0.296,1
7,LR,D+CAL+COMP,D,0.006,-0.019,0.033,0.887,2
8,LR,FULL,D,-0.027,-0.074,0.019,0.642,1
9,LR,B,A,-0.025,-0.050,-0.002,0.296,0


## 3. Nested Optuna, one worked example
For each outer fold the search only sees that fold's training window and scores trials with an inner expanding split. 8 trials here to keep the notebook quick; the saved results used 20 (12 for CatBoost).

In [4]:
y = dev.hit.values
oof, folds, info = T.nested_optuna_oof(dev, y, 'XGB', 'B3', n_trials=8, n_jobs=1)
ok = ~np.isnan(oof)
print('tuned XGB / set B3: out-of-fold AUC', round(M.auc(y[ok], oof[ok]), 3))
pd.DataFrame([{'fold': i+1, 'inner_auc': r['inner_auc'], **{k.replace('model__',''): v for k, v in r['params'].items()}} for i, r in enumerate(info)]).round(3)

tuned XGB / set B3: out-of-fold AUC 0.723


,fold,inner_auc,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda
0,1,0.713,270,0.011,4,2.535,0.626,0.974,8.536
1,2,0.707,270,0.011,4,2.535,0.626,0.974,8.536
2,3,0.698,270,0.011,4,2.535,0.626,0.974,8.536
3,4,0.687,270,0.011,4,2.535,0.626,0.974,8.536
4,5,0.686,270,0.011,4,2.535,0.626,0.974,8.536


## 4. Saved result: tuning versus defaults, and ensembles
Tuning never beat the default settings convincingly, and no ensemble convincingly beat the best single model.

In [5]:
display(pd.read_csv(C.OUT_DIR/'phase45'/'4_tuned_vs_default.csv').round(3))
display(pd.read_csv(C.OUT_DIR/'phase45'/'5_ensembles.csv').round(3).sort_values(['set','auc'], ascending=[True, False]).head(16))

,model,set,auc_default,auc_tuned,diff,ci_low,ci_high,p,mean_inner_auc,q_bh
0,XGB,A,0.681,0.669,-0.012,-0.036,0.013,0.371,0.707,0.464
1,LGBM,A,0.678,0.671,-0.007,-0.033,0.018,0.582,0.696,0.647
2,CAT,A,0.653,0.682,0.029,-0.001,0.060,0.060,0.679,0.300
3,XGB,B,0.707,0.683,-0.024,-0.051,0.004,0.097,0.709,0.323
4,LGBM,B,0.703,0.684,-0.018,-0.049,0.012,0.239,0.698,0.386
5,CAT,B,0.712,0.700,-0.012,-0.035,0.009,0.270,0.684,0.386
6,XGB,B3,0.718,0.690,-0.028,-0.053,-0.002,0.029,0.708,0.290
7,LGBM,B3,0.710,0.692,-0.018,-0.048,0.014,0.267,0.702,0.386
8,XGB,D,0.709,0.706,-0.003,-0.031,0.027,0.839,0.712,0.839
9,LGBM,D,0.701,0.683,-0.018,-0.047,0.011,0.230,0.698,0.386


,set,predictor,kind,auc,pr_auc,brier,diff_vs_best_single,ci_low,ci_high,p,best_single
5,A,average,ensemble,0.688,0.694,0.220,0.003,-0.016,0.022,0.740,HGB
6,A,weighted_vote,ensemble,0.687,0.694,0.220,0.003,-0.016,0.021,0.810,HGB
1,A,HGB,single,0.685,0.690,0.222,0.000,0.000,0.000,1.000,HGB
2,A,tuned_XGB,single,0.680,0.684,0.223,-0.005,-0.030,0.019,0.696,HGB
0,A,LR,single,0.679,0.666,0.226,-0.005,-0.043,0.035,0.760,HGB
3,A,tuned_LGBM,single,0.679,0.687,0.223,-0.006,-0.035,0.020,0.668,HGB
4,A,tuned_CAT,single,0.675,0.675,0.224,-0.010,-0.036,0.015,0.384,HGB
7,A,stacking,ensemble,0.675,0.656,0.227,-0.009,-0.033,0.014,0.396,HGB
13,B,average,ensemble,0.710,0.700,0.215,0.011,-0.011,0.032,0.362,tuned_XGB
14,B,weighted_vote,ensemble,0.710,0.701,0.215,0.011,-0.012,0.032,0.358,tuned_XGB


## 5. Calibration (fitted only on earlier folds)
The Hit rate drifts down over time, so calibrators learned on earlier folds make probabilities worse, not better. The deployed models are therefore not recalibrated.

In [6]:
pd.read_csv(C.OUT_DIR/'phase45'/'5_calibration.csv').round(3)

,calibration,predictor,auc,pr_auc,brier,log_loss,ece
0,none,tuned_LGBM/B3,0.720,0.710,0.215,0.619,0.071
1,platt,tuned_LGBM/B3,0.726,0.699,0.237,0.667,0.138
2,isotonic,tuned_LGBM/B3,0.709,0.647,0.226,0.644,0.097


## 6. Final test (scored once)

In [7]:
display(pd.read_csv(C.OUT_DIR/'final'/'test_results.csv').iloc[:, :11].round(3))
pd.read_csv(C.OUT_DIR/'final'/'test_comparisons.csv').round(3)

,model,auc,auc_ci_low,auc_ci_high,pr_auc,pr_ci_low,pr_ci_high,brier,log_loss,ece,mean_predicted
0,"basic (HGB, set A)",0.735,0.667,0.802,0.562,0.455,0.685,0.186,0.559,0.112,0.395
1,"buzz (XGB, set B3)",0.776,0.717,0.834,0.571,0.463,0.692,0.173,0.518,0.094,0.341
2,"reference: LR, set A",0.759,0.692,0.822,0.601,0.494,0.716,0.175,0.531,0.105,0.346
3,"reference: XGB, set D (earlier headline)",0.760,0.697,0.819,0.549,0.446,0.676,0.178,0.534,0.090,0.353
4,dummy (always the development Hit rate),0.500,0.500,0.500,0.297,NaN,NaN,0.239,0.671,0.179,0.471


,model,vs,auc_diff,ci_low,ci_high,p,q_bh
0,"basic (HGB, set A)","reference: LR, set A",-0.024,-0.072,0.022,0.324,0.648
1,"buzz (XGB, set B3)","reference: LR, set A",0.017,-0.036,0.071,0.499,0.665
2,"reference: XGB, set D (earlier headline)","reference: LR, set A",0.001,-0.056,0.059,0.952,0.952
3,"buzz (XGB, set B3)","basic (HGB, set A)",0.041,-0.010,0.096,0.123,0.492
